# DDoS Attack Detection: Exploratory Data Analysis

This notebook provides exploratory analysis on the flooding-based DDoS multi-class dataset used to train the machine learning detection engine.

### Objectives:
- Inspect dataset schema, data types, and missing values
- Analyze class balance across Normal and DDoS flooding attack categories
- Examine distribution of network flow features (flags, packet sizes, inter-arrival times)
- Understand feature separability for model training

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Cleaned Dataset

In [ ]:
DATA_PATH = Path("../data/processed/ddos_cleaned.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../data/ddos_cleaned.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()

## 2. Target Class Distribution

In [ ]:
class_counts = df["Label"].value_counts()
print("Class Breakdown:")
print(class_counts)

plt.figure(figsize=(9, 4))
sns.barplot(x=class_counts.values, y=class_counts.index, palette="viridis")
plt.title("Class Distribution in DDoS Dataset", fontsize=14)
plt.xlabel("Sample Count")
plt.ylabel("Traffic Type")
plt.tight_layout()
plt.show()

## 3. Protocol and Flag Statistics
Different DDoS attacks exhibit distinct flag patterns (e.g. SYN Flooding vs ACK Flooding).

In [ ]:
flag_cols = ["Avg_syn_flag", "Avg_ack_flag", "Avg_fin_flag", "Avg_rst_flag", "Avg_psh_flag", "Avg_urg_flag"]
flag_summary = df.groupby("Label")[flag_cols].mean()
print("Mean Flag Activation by Traffic Class:")
print(flag_summary)

## 4. Flow Timing & Packet Length Comparison

In [ ]:
timing_cols = ["Avg_delta_time", "Avg_pkts_lenght", "Avg_payload"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, timing_cols):
    sns.boxplot(data=df, x="Label", y=col, ax=ax, palette="Set2")
    ax.set_title(f"Distribution of {col}")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 5. Summary & Next Steps
- The dataset features clear discriminative characteristics between Normal and attack flows.
- Flag averages provide sharp signals for SYN/ACK flooding.
- Model training (`src/train_models.py`) and live feature extraction (`src/feature_extractor.py`) align directly with these statistics.